# 게임 추천 모델 비교 (현빈 담당)

4개 모델 — **인기도 / 콘텐츠 / User-CF / Item-CF** — 를 같은 NDCG 기준으로 비교한다.

- 데이터: `data/processed/cleaned_played_data.csv` (0시간 제거), `game_features_full.csv`, 취향벡터 2개
- 평가: hold-out(유저 게임 30% 숨김) → **NDCG@10 / Recall@10**, 인기도 baseline 대비
- 모델/평가 로직은 `src/` 에 모듈로 분리 (가윤의 SVD·MAB도 같은 인터페이스로 추가 가능)

## 1. 셋업 & 데이터 로드

In [ ]:
import sys
sys.path.insert(0, "../src")   # 노트북이 notebooks/ 에 있을 때
import data, evaluation
from models import MODELS

interactions, game, ug, ut = data.load_raw("../data/processed")
# 팀 공식 유저(779명)로 제한
interactions = interactions[interactions["steamid"].isin(set(ug["steamid"]))]
print("상호작용:", len(interactions), "| 유저:", interactions["steamid"].nunique(),
      "| 게임:", interactions["appid"].nunique())

## 2. train / test 분할 (hold-out)

각 유저의 플레이 게임 30%를 숨겨 정답(test)으로 두고, 나머지 70%로만 모델을 만든다 (데이터 누수 방지).

In [ ]:
train, test = evaluation.train_test_split(interactions, test_ratio=0.3, min_games=8, seed=42)
print("학습 상호작용:", len(train), "| 평가 유저:", len(test))

## 3. 공통 재료(ctx) 준비

train 상호작용만으로 행렬·취향벡터·게임벡터·인기도·유사도를 만든다.

In [ ]:
ctx = data.build_context(train, game, ug, ut)
print("유저×게임 행렬:", ctx["matrix"].shape)
print("게임 벡터:", ctx["game_vectors"].shape, "(42차원 장르+태그)")
print("준비된 재료:", list(ctx.keys()))

## 4. 4개 모델 NDCG 비교

모든 모델을 같은 test로 채점한다. `MODELS` 딕셔너리에 함수만 추가하면 비교에 합류.

In [ ]:
table = evaluation.evaluate(MODELS, ctx, test, k=10)
table.round(4)

## 5. 특정 유저 추천 예시 (정성 확인)

In [ ]:
import pandas as pd
uid = list(test.keys())[0]
played = ctx["interactions"]
top_played = played[played["steamid"]==uid].nlargest(5, "w")["game_name"].tolist()
print("유저", uid, "가 많이 한 게임:", top_played)
print()
for name, fn in MODELS.items():
    scores = fn(uid, ctx)
    scores = scores.drop(index=[a for a in ctx["owned"].get(uid,set()) if a in scores.index], errors="ignore")
    top = scores.nlargest(5).index
    print(f"[{name}]", [str(ctx["names"].get(a, a)) for a in top])

## 6. 결과 저장

In [ ]:
import os
os.makedirs("../results", exist_ok=True)
table.to_csv("../results/model_comparison.csv", encoding="utf-8-sig")
print("저장: results/model_comparison.csv")
table.round(4)

## 메모 (개선 실험 후보)
- **콘텐츠 Value Score 가격(δ)항 추가** 테스트 → NDCG 오르나 확인
- 하이브리드(콘텐츠+CF) 가중치 튜닝
- `recent_playtime_hours`(최근 2주)로 현재 취향 반영
- 분할 seed 여러 개 평균(교차검증)으로 숫자 안정화

## 팀 통합
`MODELS`에 가윤의 `svd`, `mab` 함수를 같은 `recommend(user, ctx)` 형식으로 추가하면
6개 모델을 이 노트북 하나로 같은 기준(NDCG@10)으로 비교할 수 있다.